# MMMU-Pro Eval300 × InternVL3.5-8B Q4_K_M — P4 Generated Knowledge Prompting

**P4 = zero-shot Generated Knowledge Prompting (GKP), 3 generated facts,
answer-token log probabilities, and max-probability aggregation.**

The notebook intentionally has two resumable phases because a Kaggle GPU
session is time-limited:

1. **KNOWLEDGE** — generate exactly three facts per Eval300 sample and freeze them.
2. **INFERENCE** — in a later session, evaluate the same question three times,
   once with each frozen fact, request option-token log probabilities, and
   select the answer supported by the single largest probability.

The answer stage sends **no demonstrations** to the model.


## Frozen P4 protocol

Knowledge generation follows the GKP structure: the prompt contains an
instruction, three same-subject multimodal question→knowledge demonstrations, then the
new target input. The same prompt is sampled three times with fixed stochastic
seeds.

For the XLSX demonstration bank, this notebook uses the **first three rows of
the target subject in spreadsheet order**. No retrieval, embeddings, cosine
similarity, or difficulty selection is performed.

Knowledge-generation settings:
- `do_sample`: operationally enabled by non-zero temperature + nucleus sampling
- `top_p = 0.5`
- `temperature = 1.0 (InternVL3.5 model configuration default temperature)`
- max knowledge length: 128 tokens
- stop at newline
- reproducible stochastic seeds: 42, 43, 44

Knowledge integration is zero-shot. For each fact, only target image(s),
generated knowledge, question, and options are sent. The model explains its
reasoning and must end with the exact final line `Answer: $LETTER`.
Temperature is 0, seed 42, top-p/top-k are omitted, and `logprobs` are requested
at the final answer-letter position after the visible reasoning.


### Phase 1 prompt source

The textual knowledge-generation prompt now follows **Table 8 (CSQA)** of
Liu et al., *Generated Knowledge Prompting for Commonsense Reasoning*:

```text
Generate some knowledge about the concepts in the input. Examples:
Input: {demo question}
Knowledge: {demo knowledge}
...
Input: {target question}
Knowledge:
```

The only multimodal adaptation is insertion of the corresponding image(s)
immediately before each `Input:` block. The project still uses the first
three XLSX demonstrations of the exact subject.

**Phase-1 image policy:** demonstration examples include only images referenced in the demonstration question text; option-image occurrences are excluded. The target query includes images referenced in both the question and answer options. Round 2 also keeps all target image occurrences.


## Required Kaggle inputs

Attach:
- the same frozen `selected_ids.txt` used for Eval300;
- `gkp_30_subjects_from_cot_completed_149.xlsx`;
- the local model/projector/runtime input used in the preceding experiments;
- in **INFERENCE** phase, the saved Kaggle output from the completed
  **KNOWLEDGE** phase.

Internet is needed only when the pinned Hugging Face dataset revisions are not
already cached.

**Important:** change only `P4_PHASE` between the two sessions. All signatures
protect against mixing facts or inference checkpoints from a different model or
protocol.


In [1]:
# Install only missing runtime packages.
import importlib.util, subprocess, sys
required = {
    "datasets": "datasets>=3.0.0",
    "PIL": "Pillow>=10.0.0",
    "requests": "requests>=2.31.0",
    "pandas": "pandas>=2.0.0",
    "numpy": "numpy>=1.26.0",
}
missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Package gate: PASS")


Package gate: PASS


In [2]:
import os, io, re, ast, json, math, time, base64, random, shutil, hashlib
import zipfile, xml.etree.ElementTree as ET
import subprocess, threading, shlex, logging
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset

SESSION_STARTED_MONOTONIC = time.monotonic()


In [3]:
# ========================= USER SWITCH =========================
# First Kaggle session: leave "KNOWLEDGE".
# After the knowledge bank is complete, Save Version, attach that output to a
# fresh GPU session, change this one line to "INFERENCE", and Run All.
P4_PHASE = "INFERENCE"
# ===============================================================

if P4_PHASE not in {"KNOWLEDGE", "INFERENCE"}:
    raise ValueError(f"Unsupported P4_PHASE: {P4_PHASE!r}")

MODEL_KIND = "internvl"
MODEL = "InternVL3.5-8B-Q4_K_M"
MODEL_API_NAME = "internvl3.5-8b"
BASE_MODEL = "OpenGVLab/InternVL3_5-8B"
MODEL_REPO = "lmstudio-community/InternVL3_5-8B-GGUF"
MODEL_FILENAME = "InternVL3_5-8B-Q4_K_M.gguf"
MMPROJ_FILENAME = "mmproj-model-f16.gguf"
MODEL_SHA256 = "2809043479b8d3aab30378766c7a2a4bd93eedd97c86efc6d65d627fd680faba"
MMPROJ_SHA256 = "212cc090f81ea2981b870186d4b424fae69489a5313a14e52ffdb2e877852389"
MODEL_ARTIFACT_REVISION = "ae2ec0fbf3e4abb20a33e9d682c5ebf33f638ab7"
MMPROJ_ARTIFACT_REVISION = "98b41ac52d1cfb464694a45faab81ab685ad2dfb"
LOCAL_MODEL_DIRNAME = "internvl35_8b_saved"

LLAMA_CPP_COMMIT = "50f068fff"
LLAMA_RUNTIME_SHA256 = "29cf4dddaad7d8249546e1258e553e12435749345db37905a5dd0c5c35e3e36f"

DATASET_REPO = "MMMU/MMMU_Pro"
DATASET_CONFIG = "standard (10 options)"
DATASET_SPLIT = "test"
DATASET_REVISION = "563f3e84bb3b90893083a1f039cfa13077f2302b"
EXPECTED_SOURCE_N = 1730
EXPECTED_N = 300
EXPECTED_SELECTED_IDS_CANONICAL_SHA256 = "db7ec6dca5dff71d8ea8551a08c448c4314e153509ce9c78d2e8c652011a5dc3"

DEMO_REPO = "MMMU/MMMU"
DEMO_REVISION = "98e6ac0cb9b7b2cd2c991b85a50762edc4aedc68"

GKP_XLSX_NAME = "gkp_30_subjects_from_cot_completed_149.xlsx"
GKP_XLSX_SHA256 = "9f3c6f40b01f5ceabc2ed5dd37e7669ac724c603a68022cd505d9e66ac8fe290"
GKP_SHEET_NAME = "GKP_Demonstrations"
KNOWLEDGE_DEMOS_PER_SUBJECT = 3
N_KNOWLEDGE = 3

# GKP knowledge-generation sampling.
# Original GKP uses nucleus sampling p=0.5; temperature is model-specific here.
KNOWLEDGE_TEMPERATURE = 1.0
KNOWLEDGE_TOP_P = 0.5
KNOWLEDGE_MAX_TOKENS = 128
KNOWLEDGE_SEEDS = [42, 43, 44]

# Zero-shot knowledge-integration stage: baseline generation settings.
BASELINE_SEED = 42
BASELINE_MAX_TOKENS = 8192
LOGPROBS_TOP_N = 20

CTX_SIZE = 16384
TENSOR_SPLIT = "1,1"
PORT = 8080
REQUEST_TIMEOUT_S = 1800
SERVER_LOAD_TIMEOUT_S = 1200
MAX_ATTEMPTS = 2
BASE_BACKOFF_S = 5
MAX_BACKOFF_S = 60
MAX_TOTAL_SERVER_RESTARTS = 8

# Leave margin before Kaggle's hard session cutoff.
SESSION_SOFT_STOP_S = 9 * 3600

INPUT_ROOT = Path("/kaggle/input")
WORKDIR = Path(f"/kaggle/working/{MODEL_KIND}_p4_gkp_3facts_logprob")
WORKDIR.mkdir(parents=True, exist_ok=True)

EVENTS_JSONL = WORKDIR / "p4_events.jsonl"
KNOWLEDGE_RAW_JSONL = WORKDIR / "p4_knowledge_raw.jsonl"
KNOWLEDGE_BANK_JSONL = WORKDIR / "p4_knowledge_bank.jsonl"
KNOWLEDGE_MANIFEST_JSON = WORKDIR / "p4_knowledge_manifest.json"
INFERENCE_RAW_JSONL = WORKDIR / "p4_inference_raw.jsonl"
INFERENCE_CALLS_CSV = WORKDIR / "p4_inference_calls_reasoning_logprobs.csv"
FINAL_RESULTS_CSV = WORKDIR / "p4_results.csv"
SUMMARY_JSON = WORKDIR / "p4_summary.json"
README_PATH = WORKDIR / "README_RESULTS.md"

EXPERIMENT_ID = f"MMMUPro-Eval300__{MODEL}__P4-GKP-3Facts-LogProb-Max"
PROMPT_ID = "P4"
PROMPT_FAMILY = "zero-shot generated-knowledge prompting with 3 stochastic knowledge samples and max-probability integration"

print("P4 phase:", P4_PHASE)
print("Knowledge temperature:", KNOWLEDGE_TEMPERATURE)
print("Knowledge top_p:", KNOWLEDGE_TOP_P)
print("Knowledge seeds:", KNOWLEDGE_SEEDS)
print("Inference temperature:", 0.0)
print("Inference top_p/top_k:", None, None)


P4 phase: INFERENCE
Knowledge temperature: 1.0
Knowledge top_p: 0.5
Knowledge seeds: [42, 43, 44]
Inference temperature: 0.0
Inference top_p/top_k: None None


In [4]:
def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha_file(path, chunk_size=16 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def canonical_id_text(ids):
    return "\n".join(str(x) for x in ids) + "\n"


def canonical_id_sha256(ids):
    return hashlib.sha256(canonical_id_text(ids).encode("utf-8")).hexdigest()


def canonical_json_sha256(obj):
    return hashlib.sha256(
        json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")
    ).hexdigest()


def atomic_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)


def atomic_json(path, payload):
    atomic_text(path, json.dumps(payload, ensure_ascii=False, indent=2))


def append_jsonl(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    out = []
    with path.open("r", encoding="utf-8") as handle:
        for line_no, line in enumerate(handle, 1):
            line = line.strip()
            if not line:
                continue
            try:
                out.append(json.loads(line))
            except Exception as exc:
                raise RuntimeError(f"Malformed JSONL {path}:{line_no}: {exc}") from exc
    return out


def discover_named(name):
    out = []
    for root in [INPUT_ROOT, WORKDIR]:
        if not Path(root).exists():
            continue
        try:
            out.extend(p.resolve() for p in Path(root).rglob(name) if p.is_file())
        except Exception:
            pass
    return sorted(set(out), key=str)


def read_id_file_strict(path):
    text = Path(path).read_text(encoding="utf-8")
    rows = text.splitlines()
    require(rows, f"Empty ID file: {path}")
    require(not any((not x) or x != x.strip() for x in rows), f"Malformed ID file: {path}")
    return rows


def parse_options(value):
    if isinstance(value, (list, tuple)):
        return [str(x) for x in value]
    if isinstance(value, np.ndarray):
        return [str(x) for x in value.tolist()]
    if isinstance(value, str):
        for parser in (json.loads, ast.literal_eval):
            try:
                parsed = parser(value)
                if isinstance(parsed, (list, tuple)):
                    return [str(x) for x in parsed]
            except Exception:
                pass
    raise TypeError(f"Unsupported options value: {type(value)}")


def choice_letters(n):
    require(1 <= int(n) <= 26, f"Unsupported option count: {n}")
    return [chr(ord("A") + i) for i in range(int(n))]


def format_options(options):
    options = parse_options(options)
    return "\n".join(f"{letter}. {text}" for letter, text in zip(choice_letters(len(options)), options))


IMAGE_MARKER_RE = re.compile(r"<image\s*(\d+)>", flags=re.IGNORECASE)


def normalize_image_markers(text):
    return IMAGE_MARKER_RE.sub("<image>", str(text))


def image_occurrences_from_row(row, include_options=True):
    texts = [str(row.get("question") or "")]
    if include_options:
        texts.extend(parse_options(row.get("options")))
    occurrences = []
    for text in texts:
        for m in IMAGE_MARKER_RE.finditer(text):
            col = f"image_{int(m.group(1))}"
            img = row.get(col)
            if img is not None:
                occurrences.append((col, img))
    # Defensive fallback for rows whose image markers are absent/malformed.
    if not occurrences:
        image_cols = sorted(
            [k for k in row.keys() if re.fullmatch(r"image_\d+", str(k)) and row.get(k) is not None],
            key=lambda x: int(x.split("_")[1]),
        )
        occurrences = [(c, row[c]) for c in image_cols]
    return occurrences


def pil_to_png_bytes(image):
    if image is None:
        raise ValueError("Image is None")
    if not isinstance(image, Image.Image):
        # HF Image feature may occasionally return a mapping containing bytes/path.
        if isinstance(image, dict) and image.get("bytes"):
            image = Image.open(io.BytesIO(image["bytes"]))
        elif isinstance(image, dict) and image.get("path"):
            image = Image.open(image["path"])
        else:
            raise TypeError(f"Unsupported image object: {type(image)}")
    image = image.convert("RGB")
    buf = io.BytesIO()
    image.save(buf, format="PNG", optimize=False)
    return buf.getvalue()


def png_part(label, image_col, image):
    data = pil_to_png_bytes(image)
    digest = hashlib.sha256(data).hexdigest()
    return {
        "type": "image",
        "label": str(label),
        "image_col": str(image_col),
        "sha256": digest,
        "png_bytes": len(data),
        "data_url": "data:image/png;base64," + base64.b64encode(data).decode("ascii"),
    }


def openai_content(parts):
    content = []
    for part in parts:
        if part["type"] == "text":
            content.append({"type": "text", "text": part["text"]})
        elif part["type"] == "image":
            content.append({"type": "image_url", "image_url": {"url": part["data_url"]}})
        else:
            raise ValueError(part["type"])
    return content


def canonical_parts_text(parts):
    chunks = []
    for p in parts:
        if p["type"] == "text":
            chunks.append(p["text"])
        else:
            chunks.append(f"<image:{p['sha256']}>")
    return "".join(chunks)


def session_soft_stop_reached():
    return (time.monotonic() - SESSION_STARTED_MONOTONIC) >= SESSION_SOFT_STOP_S


def logsumexp(values):
    values = [float(x) for x in values]
    if not values:
        return float("-inf")
    m = max(values)
    return m + math.log(sum(math.exp(x - m) for x in values))


# Minimal standard-library XLSX reader.
# This avoids an openpyxl dependency in Kaggle and preserves sheet row order.
def read_xlsx_sheet(path, sheet_name):
    path = Path(path)
    with zipfile.ZipFile(path, "r") as zf:
        ns = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
              "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
              "p": "http://schemas.openxmlformats.org/package/2006/relationships"}

        shared = []
        if "xl/sharedStrings.xml" in zf.namelist():
            root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
            for si in root.findall("m:si", ns):
                texts = [t.text or "" for t in si.findall(".//m:t", ns)]
                shared.append("".join(texts))

        wb_root = ET.fromstring(zf.read("xl/workbook.xml"))
        rel_root = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
        rel_map = {
            rel.attrib["Id"]: rel.attrib["Target"]
            for rel in rel_root.findall("p:Relationship", ns)
        }

        target = None
        for sh in wb_root.findall("m:sheets/m:sheet", ns):
            if sh.attrib.get("name") == sheet_name:
                rid = sh.attrib.get("{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id")
                target = rel_map[rid]
                break
        require(target is not None, f"Sheet not found: {sheet_name}")
        if not target.startswith("/"):
            sheet_path = "xl/" + target.lstrip("/")
        else:
            sheet_path = target.lstrip("/")

        root = ET.fromstring(zf.read(sheet_path))
        rows = []
        for row in root.findall(".//m:sheetData/m:row", ns):
            values = {}
            for c in row.findall("m:c", ns):
                ref = c.attrib["r"]
                letters = re.match(r"[A-Z]+", ref).group(0)
                col = 0
                for ch in letters:
                    col = col * 26 + (ord(ch) - 64)
                col -= 1
                ctype = c.attrib.get("t")
                if ctype == "inlineStr":
                    txt = "".join(t.text or "" for t in c.findall(".//m:t", ns))
                    val = txt
                else:
                    v = c.find("m:v", ns)
                    raw = None if v is None else v.text
                    if raw is None:
                        val = None
                    elif ctype == "s":
                        val = shared[int(raw)]
                    elif ctype == "b":
                        val = bool(int(raw))
                    else:
                        try:
                            f = float(raw)
                            val = int(f) if f.is_integer() else f
                        except Exception:
                            val = raw
                values[col] = val
            if values:
                width = max(values) + 1
                rows.append([values.get(i) for i in range(width)])
        require(rows, f"No rows in sheet {sheet_name}")
        headers = [str(x) if x is not None else "" for x in rows[0]]
        output = []
        for row in rows[1:]:
            padded = row + [None] * (len(headers) - len(row))
            rec = dict(zip(headers, padded[:len(headers)]))
            if any(v is not None for v in rec.values()):
                output.append(rec)
        return output


## Verify the SHA-pinned local InternVL3.5-8B Q4_K_M model, projector, CUDA driver, and llama runtime

In [5]:
# Exact InternVL model/projector pair from the user's Kaggle Input.
model_candidates = sorted([
    p.resolve() for p in INPUT_ROOT.rglob(MODEL_FILENAME)
    if p.is_file() and p.parent.name == LOCAL_MODEL_DIRNAME
], key=str)
verified_models, diagnostics = [], []
for model_path in model_candidates:
    mmproj_path = model_path.parent / MMPROJ_FILENAME
    rec = {"model": str(model_path), "mmproj": str(mmproj_path)}
    try:
        rec["model_sha"] = sha_file(model_path)
        rec["mmproj_sha"] = sha_file(mmproj_path) if mmproj_path.is_file() else None
        if rec["model_sha"] == MODEL_SHA256 and rec["mmproj_sha"] == MMPROJ_SHA256:
            verified_models.append((model_path, mmproj_path)); rec["status"] = "accepted"
        else:
            rec["status"] = "rejected"
    except Exception as exc:
        rec["error"] = repr(exc)
    diagnostics.append(rec)
require(verified_models, "No exact InternVL3.5-8B model/projector pair found. Expected internvl35_8b_saved/InternVL3_5-8B-Q4_K_M.gguf and internvl35_8b_saved/mmproj-model-f16.gguf. Diagnostics:\n" + json.dumps(diagnostics, indent=2))
MODEL_PATH, MMPROJ_PATH = sorted(verified_models, key=lambda x: str(x[0]))[0]

# Runtime may be attached separately. Accept only the exact previously validated binary.
runtime_candidates = sorted(set(p.resolve() for p in INPUT_ROOT.rglob("llama") if p.is_file()), key=str)
verified_runtimes, runtime_diag = [], []
for p in runtime_candidates:
    try:
        digest = sha_file(p); runtime_diag.append({"path": str(p), "sha256": digest})
        if digest == LLAMA_RUNTIME_SHA256: verified_runtimes.append(p)
    except Exception as exc:
        runtime_diag.append({"path": str(p), "error": repr(exc)})
require(verified_runtimes, "No exact saved llama runtime found for InternVL. Attach the previously validated runtime SHA256 " + LLAMA_RUNTIME_SHA256 + ". Diagnostics:\n" + json.dumps(runtime_diag[:100], indent=2))
INPUT_LLAMA_BIN = verified_runtimes[0]
MODEL_SHA_OBSERVED = sha_file(MODEL_PATH); MMPROJ_SHA_OBSERVED = sha_file(MMPROJ_PATH)
require(MODEL_SHA_OBSERVED == MODEL_SHA256, "Model SHA mismatch")
require(MMPROJ_SHA_OBSERVED == MMPROJ_SHA256, "Projector SHA mismatch")
require(sha_file(INPUT_LLAMA_BIN) == LLAMA_RUNTIME_SHA256, "Input llama SHA mismatch")

# GPU gate BEFORE launching the CUDA-linked runtime.
NVIDIA_SMI = shutil.which("nvidia-smi")
require(NVIDIA_SMI is not None, "Kaggle GPU is unavailable. Enable Accelerator = T4 x2 and restart the session.")
smi = subprocess.run([NVIDIA_SMI, "--query-gpu=index,name,memory.total,compute_cap", "--format=csv,noheader"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=False)
require(smi.returncode == 0 and (smi.stdout or "").strip(), "nvidia-smi cannot access the GPU driver:\n" + (smi.stdout or "")[-8000:])
gpu_lines = [x.strip() for x in smi.stdout.splitlines() if x.strip()]
gpu_names, compute_caps = [], []
for line in gpu_lines:
    cols = [x.strip() for x in line.split(",")]; require(len(cols) >= 4, f"Unexpected nvidia-smi row: {line}")
    gpu_names.append(cols[1]); compute_caps.append(float(cols[3]))
require(len(gpu_names) == 2 and all("T4" in x.upper() for x in gpu_names), f"Frozen local protocol requires T4 x2; got {gpu_names}")
require(compute_caps == [7.5, 7.5], f"Expected compute capability [7.5, 7.5]; got {compute_caps}")

def real_libcuda_dirs():
    candidates = []
    for x in os.environ.get("LD_LIBRARY_PATH", "").split(":"):
        if x.strip(): candidates.append(Path(x.strip()))
    candidates += [Path("/usr/lib/x86_64-linux-gnu"), Path("/lib/x86_64-linux-gnu"), Path("/usr/local/nvidia/lib64"), Path("/usr/local/nvidia/lib")]
    ldconfig = shutil.which("ldconfig")
    if ldconfig:
        p = subprocess.run([ldconfig, "-p"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, check=False)
        if p.returncode == 0:
            for line in p.stdout.splitlines():
                if "libcuda.so.1" in line and "=>" in line: candidates.append(Path(line.split("=>",1)[1].strip()).parent)
    out, seen = [], set()
    for d in candidates:
        try: d=d.resolve()
        except Exception: continue
        if str(d) in seen or "/stubs" in str(d): continue
        seen.add(str(d))
        if (d/"libcuda.so.1").exists(): out.append(d)
    return out

CUDA_DRIVER_LIBRARY_DIRS = real_libcuda_dirs()
require(CUDA_DRIVER_LIBRARY_DIRS, "T4 is visible but libcuda.so.1 is not visible inside this process. Restart the Kaggle T4 x2 session.")
RUNTIME_ENV = dict(os.environ); RUNTIME_ENV["CUDA_VISIBLE_DEVICES"] = "0,1"
existing_ld=[x for x in RUNTIME_ENV.get("LD_LIBRARY_PATH","").split(":") if x]
RUNTIME_ENV["LD_LIBRARY_PATH"] = ":".join(dict.fromkeys([str(x) for x in CUDA_DRIVER_LIBRARY_DIRS]+existing_ld))

# Never execute the Kaggle Input runtime directly (read-only/noexec risk).
RUNTIME_DIR = Path("/kaggle/temp/internvl35_8b_p4_runtime"); RUNTIME_DIR.mkdir(parents=True,exist_ok=True)
LLAMA_BIN = RUNTIME_DIR/"llama"; shutil.copy2(INPUT_LLAMA_BIN, LLAMA_BIN); os.chmod(LLAMA_BIN,0o755)
require(LLAMA_BIN.is_file() and os.access(LLAMA_BIN,os.X_OK), "Writable llama runtime is not executable")
require(sha_file(LLAMA_BIN)==LLAMA_RUNTIME_SHA256, "Copied llama runtime SHA mismatch")
require(not str(LLAMA_BIN.resolve()).startswith("/kaggle/input/"), "Never execute llama from /kaggle/input")
ldd=shutil.which("ldd")
if ldd:
    chk=subprocess.run([ldd,str(LLAMA_BIN)],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,check=False,env=RUNTIME_ENV)
    missing_shared_libs=[line.strip() for line in (chk.stdout or "").splitlines() if "=> not found" in line]
    require(not missing_shared_libs, "Unresolved llama shared libraries:\n"+"\n".join(missing_shared_libs))
else: missing_shared_libs=[]

def run_capture(cmd,timeout=180):
    exe=Path(cmd[0]).resolve(); require(not str(exe).startswith("/kaggle/input/"),f"Refusing execution from /kaggle/input: {exe}")
    require(exe.is_file() and os.access(exe,os.X_OK),f"Runtime executable gate failed: {exe}")
    if exe==LLAMA_BIN: require(sha_file(exe)==LLAMA_RUNTIME_SHA256,"Runtime SHA drift before subprocess")
    proc=subprocess.run([str(x) for x in cmd],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=timeout,check=False,env=RUNTIME_ENV)
    require(proc.returncode==0,f"Command failed rc={proc.returncode}: {cmd}\n{(proc.stdout or '')[-12000:]}")
    return proc.stdout

LLAMA_RUNTIME_VERSION_TEXT=run_capture([str(LLAMA_BIN),"version"]).strip()
require(("10679" in LLAMA_RUNTIME_VERSION_TEXT) or (LLAMA_CPP_COMMIT in LLAMA_RUNTIME_VERSION_TEXT),"Unexpected llama runtime build: "+LLAMA_RUNTIME_VERSION_TEXT[-3000:])
device_text=run_capture([str(LLAMA_BIN),"serve","--list-devices"]).strip()
cuda_lines=[x.strip() for x in device_text.splitlines() if x.strip().lower().startswith("cuda")]
require(len(cuda_lines)>=2,f"llama runtime does not expose T4 x2: {cuda_lines}")
print("InternVL model:",MODEL_PATH); print("InternVL projector:",MMPROJ_PATH); print("Runtime input:",INPUT_LLAMA_BIN); print("Runtime exec:",LLAMA_BIN); print("Runtime version:",LLAMA_RUNTIME_VERSION_TEXT); print("CUDA driver dirs:",[str(x) for x in CUDA_DRIVER_LIBRARY_DIRS]); print("GPU:",gpu_lines)


InternVL model: /kaggle/input/notebooks/vernvern/notebookf785af4a9b/internvl35_8b_saved/InternVL3_5-8B-Q4_K_M.gguf
InternVL projector: /kaggle/input/notebooks/vernvern/notebookf785af4a9b/internvl35_8b_saved/mmproj-model-f16.gguf
Runtime input: /kaggle/input/notebooks/vernvern/notebookf785af4a9b/internvl35_llama_runtime/llama
Runtime exec: /kaggle/temp/internvl35_8b_p4_runtime/llama
Runtime version: version: 0.3.0-dev (build 10679, commit 50f068fff)
built with GNU 12.3.0 for Linux x86_64
CUDA driver dirs: ['/usr/local/nvidia/lib64']
GPU: ['0, Tesla T4, 15360 MiB, 7.5', '1, Tesla T4, 15360 MiB, 7.5']


## Load frozen Eval300 and the first-three same-subject GKP demonstration bank

In [6]:
# ------------------------------------------------------------------
# Frozen Eval300
# ------------------------------------------------------------------
selected_candidates = discover_named("selected_ids.txt")
selected_matches = []
diagnostics = []
for path in selected_candidates:
    try:
        ids = read_id_file_strict(path)
        csha = canonical_id_sha256(ids)
        diagnostics.append({"path": str(path), "rows": len(ids), "canonical_sha256": csha})
        if csha == EXPECTED_SELECTED_IDS_CANONICAL_SHA256:
            selected_matches.append((path, ids))
    except Exception as exc:
        diagnostics.append({"path": str(path), "error": repr(exc)})
require(selected_matches, "No selected_ids.txt matches Eval300. Diagnostics:\n" + json.dumps(diagnostics, indent=2))
SELECTED_IDS_PATH, selected_ids = sorted(selected_matches, key=lambda x: str(x[0]))[0]
require(len(selected_ids) == EXPECTED_N and len(set(selected_ids)) == EXPECTED_N, "Eval300 ID cardinality drift")

source_ds = load_dataset(
    DATASET_REPO, DATASET_CONFIG,
    split=DATASET_SPLIT, revision=DATASET_REVISION,
)
require(len(source_ds) == EXPECTED_SOURCE_N, f"MMMU-Pro row count drift: {len(source_ds)}")
source_index_by_id = {str(x): i for i, x in enumerate(source_ds["id"])}
missing = [x for x in selected_ids if x not in source_index_by_id]
require(not missing, f"Selected IDs missing from pinned source: {missing[:20]}")

query_meta_by_id = {}
for order, sid in enumerate(selected_ids):
    idx = source_index_by_id[sid]
    row = source_ds[idx]
    opts = parse_options(row["options"])
    letters = choice_letters(len(opts))
    gold = str(row["answer"]).strip().upper()
    require(gold in letters, f"Bad gold label {sid}: {gold}")
    query_meta_by_id[sid] = {
        "sample_id": sid,
        "dataset_index": idx,
        "execution_order": order,
        "subject": str(row["subject"]),
        "topic_difficulty": row.get("topic_difficulty"),
        "img_type": row.get("img_type"),
        "question": str(row["question"]),
        "options": opts,
        "choice_letters": letters,
        "gold_answer": gold,
    }

require([query_meta_by_id[x]["sample_id"] for x in selected_ids] == selected_ids, "Eval300 order drift")


# ------------------------------------------------------------------
# GKP knowledge demonstrations
# ------------------------------------------------------------------
gkp_candidates = [p for p in discover_named(GKP_XLSX_NAME) if sha_file(p) == GKP_XLSX_SHA256]
require(
    gkp_candidates,
    f"No {GKP_XLSX_NAME} with expected SHA-256 {GKP_XLSX_SHA256} found under Kaggle Input."
)
GKP_XLSX_PATH = sorted(gkp_candidates, key=str)[0]
gkp_rows = read_xlsx_sheet(GKP_XLSX_PATH, GKP_SHEET_NAME)

required_cols = {"subject", "question", "image", "options", "knowledge"}
require(required_cols.issubset(gkp_rows[0]), f"GKP XLSX missing columns: {required_cols - set(gkp_rows[0])}")

gkp_by_subject = defaultdict(list)
for row_number, rec in enumerate(gkp_rows, start=2):
    subject = str(rec["subject"])
    image_meta = json.loads(str(rec["image"]))
    opts = parse_options(rec["options"])
    gkp_by_subject[subject].append({
        "xlsx_row_number": row_number,
        "subject": subject,
        "question": str(rec["question"]),
        "options": opts,
        "knowledge": str(rec["knowledge"]).strip(),
        "image_meta": image_meta,
    })

query_subjects = sorted({query_meta_by_id[x]["subject"] for x in selected_ids})
require(len(query_subjects) == 30, f"Expected 30 Eval300 subjects, got {len(query_subjects)}")
missing_subjects = [s for s in query_subjects if len(gkp_by_subject.get(s, [])) < KNOWLEDGE_DEMOS_PER_SUBJECT]
require(not missing_subjects, f"GKP XLSX has fewer than 3 demos for: {missing_subjects}")

# Exact user-requested policy: FIRST THREE spreadsheet rows of the same subject.
knowledge_demo_selection = {
    subject: gkp_by_subject[subject][:KNOWLEDGE_DEMOS_PER_SUBJECT]
    for subject in query_subjects
}
KNOWLEDGE_DEMO_MAP = {
    subject: [x["image_meta"]["question_id"] for x in knowledge_demo_selection[subject]]
    for subject in query_subjects
}
KNOWLEDGE_DEMO_MAP_SHA256 = canonical_json_sha256(KNOWLEDGE_DEMO_MAP)

# Reconstruct the 90 selected MMMU demonstration rows and their images from the
# pinned MMMU revision. The XLSX row order remains authoritative.
demo_dataset_cache = {}
knowledge_demo_runtime = {}

for subject in tqdm(query_subjects, desc="Reconstructing first-3 GKP demonstrations"):
    runtime_rows = []
    for rec in knowledge_demo_selection[subject]:
        meta = rec["image_meta"]
        split = str(meta["source_split"])
        cache_key = (subject, split)
        if cache_key not in demo_dataset_cache:
            demo_dataset_cache[cache_key] = load_dataset(
                DEMO_REPO, subject, split=split, revision=DEMO_REVISION
            )
        dset = demo_dataset_cache[cache_key]
        qid = str(meta["question_id"])

        candidate_idx = meta.get("hf_local_row_in_group")
        demo_row = None
        if candidate_idx is not None:
            candidate_idx = int(candidate_idx)
            if 0 <= candidate_idx < len(dset):
                candidate = dset[candidate_idx]
                if str(candidate.get("id")) == qid:
                    demo_row = candidate
        if demo_row is None:
            matches = [j for j, x in enumerate(dset["id"]) if str(x) == qid]
            require(len(matches) == 1, f"Could not uniquely reconstruct GKP demo {qid}")
            demo_row = dset[matches[0]]

        runtime_rows.append({
            **rec,
            "dataset_row": demo_row,
        })
    knowledge_demo_runtime[subject] = runtime_rows

print("Eval300 selected IDs:", SELECTED_IDS_PATH)
print("GKP XLSX:", GKP_XLSX_PATH)
print("GKP rows:", len(gkp_rows))
print("Knowledge demo map SHA256:", KNOWLEDGE_DEMO_MAP_SHA256)
print("Policy: first 3 spreadsheet demonstrations from the exact same subject")


README.md: 0.00B [00:00, ?B/s]

standard (10 options)/test-00000-of-0000(…):   0%|          | 0.00/346M [00:00<?, ?B/s]

standard (10 options)/test-00001-of-0000(…):   0%|          | 0.00/332M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1730 [00:00<?, ? examples/s]

Reconstructing first-3 GKP demonstrations:   0%|          | 0/30 [00:00<?, ?it/s]

...

Sociology/test-00000-of-00001.parquet:   0%|          | 0.00/144M [00:00<?, ?B/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/252 [00:00<?, ? examples/s]

Eval300 selected IDs: /kaggle/input/datasets/vernvern/300-sample-mmmu-pro-dataset/mmmu_pro_eval300_seed42_reclaim_easy_hard_first/selected_ids.txt
GKP XLSX: /kaggle/input/datasets/vernvern/knowledge/gkp_30_subjects_from_cot_completed_149.xlsx
GKP rows: 149
Knowledge demo map SHA256: 77011d6a6c261aa332fb94e217aaa3f7a80edacc55b3ec216a5378789310205e
Policy: first 3 spreadsheet demonstrations from the exact same subject


## P4 prompts — few-shot only for knowledge elicitation; zero-shot for answer integration

In [7]:
KNOWLEDGE_INSTRUCTION = "Generate some knowledge about the concepts in the input. Examples:"


INFERENCE_INSTRUCTION = (
    "Answer the preceding multiple choice question.\n"
    "The last line of your response should be of the following format:\n"
    "'Answer: $LETTER' (without quotes) where LETTER is one of options.\n"
    "Explain your reasoning before answering."
)


def _append_text(parts, text):
    parts.append({"type": "text", "text": str(text)})


def _append_row_images(parts, row, prefix, include_options):
    occurrences = image_occurrences_from_row(row, include_options=include_options)
    for occ_index, (image_col, image) in enumerate(occurrences, start=1):
        _append_text(parts, f"\n[{prefix} IMAGE {occ_index}]\n")
        parts.append(png_part(f"{prefix} IMAGE {occ_index}", image_col, image))
    return len(occurrences)


def knowledge_prompt_parts(sample_id):
    qmeta = query_meta_by_id[sample_id]
    parts = []

    _append_text(parts, KNOWLEDGE_INSTRUCTION + "\n")

    for demo_index, demo in enumerate(knowledge_demo_runtime[qmeta["subject"]], start=1):
        _append_row_images(parts, demo["dataset_row"], f"DEMONSTRATION {demo_index}", include_options=False)
        demo_question = normalize_image_markers(demo["question"])
        _append_text(
            parts,
            f"Input: {demo_question}\n"
            f"Knowledge: {demo['knowledge']}\n"
        )

    row = source_ds[qmeta["dataset_index"]]
    _append_row_images(parts, row, "TARGET", include_options=True)
    target_question = normalize_image_markers(qmeta["question"])
    _append_text(parts, f"Input: {target_question}\nKnowledge:")
    return parts

def knowledge_request_payload(sample_id, fact_index):
    require(0 <= fact_index < N_KNOWLEDGE, fact_index)
    payload = {
        "model": MODEL_API_NAME,
        "messages": [{"role": "user", "content": openai_content(knowledge_prompt_parts(sample_id))}],
        "temperature": KNOWLEDGE_TEMPERATURE,
        "top_p": KNOWLEDGE_TOP_P,
        "seed": KNOWLEDGE_SEEDS[fact_index],
        "max_tokens": KNOWLEDGE_MAX_TOKENS,
        "stop": ["\n"],
        "stream": False,
    }
    return payload


def _append_target_images_round2(parts, row):
    _append_text(parts, "[TARGET IMAGE(S)]\n")
    occurrences = image_occurrences_from_row(row, include_options=True)
    for occ_index, (image_col, image) in enumerate(occurrences, start=1):
        parts.append(png_part(f"TARGET IMAGE {occ_index}", image_col, image))
    return len(occurrences)


def inference_prompt_parts(sample_id, fact):
    qmeta = query_meta_by_id[sample_id]
    row = source_ds[qmeta["dataset_index"]]
    parts = []
    _append_target_images_round2(parts, row)
    _append_text(parts, f"\nKnowledge:\n{str(fact).strip()}\n\n")
    question = normalize_image_markers(qmeta["question"])
    options = format_options([normalize_image_markers(x) for x in qmeta["options"]])
    _append_text(
        parts,
        f"{question}\n\n"
        f"{options}\n\n"
        f"{INFERENCE_INSTRUCTION}"
    )
    return parts

def inference_request_payload(sample_id, fact):
    payload = {
        "model": MODEL_API_NAME,
        "messages": [{"role": "user", "content": openai_content(inference_prompt_parts(sample_id, fact))}],
        "temperature": 0.0,
        "seed": BASELINE_SEED,
        "max_tokens": BASELINE_MAX_TOKENS,
        "stream": False,
        "logprobs": True,
        "top_logprobs": LOGPROBS_TOP_N,
    }
    return payload


# Freeze prompt identities.
KNOWLEDGE_PROMPT_TEMPLATE_SHA256 = hashlib.sha256(
    (
        "Generate some knowledge about the concepts in the input. Examples:\n"
        "{DEMO_IMAGE(S)}Input: {DEMO_QUESTION}\nKnowledge: {DEMO_KNOWLEDGE}\n"
        "{DEMO_IMAGE(S)}Input: {DEMO_QUESTION}\nKnowledge: {DEMO_KNOWLEDGE}\n"
        "{DEMO_IMAGE(S)}Input: {DEMO_QUESTION}\nKnowledge: {DEMO_KNOWLEDGE}\n"
        "{TARGET_IMAGE(S)}Input: {TARGET_QUESTION}\nKnowledge:"
    ).encode("utf-8")
).hexdigest()


INFERENCE_PROMPT_TEMPLATE_SHA256 = hashlib.sha256(
    (
        "[TARGET IMAGE(S)]\n"
        "{TARGET_IMAGE_OCCURRENCES}\n"
        "Knowledge:\n{GENERATED_KNOWLEDGE}\n\n"
        "{QUESTION}\n\n"
        "{OPTIONS}\n\n"
        "Answer the preceding multiple choice question.\n"
        "The last line of your response should be of the following format:\n"
        "'Answer: $LETTER' (without quotes) where LETTER is one of options.\n"
        "Explain your reasoning before answering."
    ).encode("utf-8")
).hexdigest()


# Static sample audit without inference.
_probe_id = selected_ids[0]
_probe_k = knowledge_prompt_parts(_probe_id)
_probe_i = inference_prompt_parts(_probe_id, "Example fact.")

require(len(knowledge_demo_runtime[query_meta_by_id[_probe_id]["subject"]]) == 3, "Knowledge-demo count drift")

_probe_k_text = canonical_parts_text(_probe_k)
require(
    _probe_k_text.startswith("Generate some knowledge about the concepts in the input. Examples:\n"),
    "Knowledge instruction is not exact GKP Table-8 wording"
)
require(_probe_k_text.count("Input: ") == 4, "Expected 3 demo Inputs + 1 target Input")
require(_probe_k_text.count("Knowledge:") == 4, "Expected 3 demo Knowledge lines + 1 target cue")
require("Do not answer" not in _probe_k_text, "Custom non-paper instruction leaked")
require("Return only" not in _probe_k_text, "Custom non-paper instruction leaked")

_probe_subject = query_meta_by_id[_probe_id]["subject"]
for _demo in knowledge_demo_runtime[_probe_subject]:
    _demo_options = format_options([normalize_image_markers(x) for x in _demo["options"]])
    require(_demo_options not in _probe_k_text, "Demo answer choices leaked into Phase 1")
_target_options = format_options([normalize_image_markers(x) for x in query_meta_by_id[_probe_id]["options"]])
require(_target_options not in _probe_k_text, "Target answer choices leaked into Phase 1")

_probe_i_text = canonical_parts_text(_probe_i)
require(_probe_i_text.startswith("[TARGET IMAGE(S)]\n"), "Round-2 prefix drift")
require("\nKnowledge:\nExample fact.\n\n" in _probe_i_text, "Round-2 Knowledge block drift")
require(INFERENCE_INSTRUCTION in _probe_i_text, "Round-2 exact suffix drift")
require("[DEMONSTRATION" not in _probe_i_text, "Round-2 must remain zero-shot")

print("Knowledge prompt template SHA256:", KNOWLEDGE_PROMPT_TEMPLATE_SHA256)
print("Inference prompt template SHA256:", INFERENCE_PROMPT_TEMPLATE_SHA256)
print("P4 inference is zero-shot: no demonstration is sent in the answer/logprob stage.")


Knowledge prompt template SHA256: 2bf246fa3aad7cfba1afac080599455e0e9c01525c250d65c47062fe9484c99b
Inference prompt template SHA256: 4523f006dffca5b877b78cce49880e1a33b4e936a0ee73d1bdfa3a1b36c92f1d
P4 inference is zero-shot: no demonstration is sent in the answer/logprob stage.


## Start the local multimodal llama.cpp endpoint

In [8]:
logger = logging.getLogger("p4")
logger.setLevel(logging.INFO)

SERVER_RESTARTS = 0
server = None
server_log = None
server_thread = None
CURRENT_SERVER_LOG = None
API = f"http://127.0.0.1:{PORT}/v1/chat/completions"


def live_log(msg):
    print(f"[{datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}] {msg}", flush=True)


def tail_text(path, n=120):
    p = Path(path) if path else None
    if p is None or not p.exists():
        return "(server log unavailable)"
    return "\n".join(p.read_text(encoding="utf-8", errors="replace").splitlines()[-n:])


def _stop_server():
    global server, server_log
    if server is not None and server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=20)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=10)
    if server_log is not None:
        try:
            server_log.flush()
            server_log.close()
        except Exception:
            pass


def start_server(reason="initial"):
    global server, server_log, server_thread, SERVER_RESTARTS, CURRENT_SERVER_LOG

    resolved = str(Path(LLAMA_BIN).resolve())
    require(not resolved.startswith("/kaggle/input/"), "Unsafe llama execution from /kaggle/input")
    require(Path(LLAMA_BIN).is_file() and os.access(LLAMA_BIN, os.X_OK), f"Runtime not executable: {LLAMA_BIN}")
    require(sha_file(LLAMA_BIN) == LLAMA_RUNTIME_SHA256, "Runtime SHA drift before server launch")

    _stop_server()
    if reason != "initial":
        SERVER_RESTARTS += 1
        require(SERVER_RESTARTS <= MAX_TOTAL_SERVER_RESTARTS, "Too many server restarts")

    suffix = "initial" if reason == "initial" else f"restart_{SERVER_RESTARTS:03d}"
    CURRENT_SERVER_LOG = WORKDIR / f"{MODEL_KIND}_p4_server_{suffix}.log"

    cmd = [
        str(LLAMA_BIN), "serve",
        "-m", str(MODEL_PATH),
        "--mmproj", str(MMPROJ_PATH),
        "--host", "127.0.0.1",
        "--port", str(PORT),
        "--alias", MODEL_API_NAME,
        "--gpu-layers", "all",
        "--split-mode", "layer",
        "--tensor-split", TENSOR_SPLIT,
        "--ctx-size", str(CTX_SIZE),
        "--parallel", "1",
        "--reasoning", "off",
        "--log-colors", "off",
        "--log-timestamps",
        "--log-prefix",
        "--log-verbosity", "3",
    ]

    live_log("SERVER COMMAND: " + " ".join(shlex.quote(str(x)) for x in cmd))
    server = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, env=RUNTIME_ENV,
    )
    server_log = CURRENT_SERVER_LOG.open("a", encoding="utf-8")

    def tee(proc, fh):
        assert proc.stdout is not None
        for line in proc.stdout:
            print(f"[llama:{suffix}] {line}", end="", flush=True)
            fh.write(line)
            fh.flush()

    server_thread = threading.Thread(target=tee, args=(server, server_log), daemon=True)
    server_thread.start()

    health = f"http://127.0.0.1:{PORT}/health"
    deadline = time.time() + SERVER_LOAD_TIMEOUT_S
    while True:
        if server.poll() is not None:
            raise RuntimeError(f"Server exited during load rc={server.returncode}\n{tail_text(CURRENT_SERVER_LOG)}")
        try:
            resp = requests.get(health, timeout=2)
            if resp.status_code == 200:
                break
        except Exception:
            pass
        if time.time() > deadline:
            _stop_server()
            raise TimeoutError(f"Server not ready within {SERVER_LOAD_TIMEOUT_S}s\n{tail_text(CURRENT_SERVER_LOG)}")
        time.sleep(2)
    live_log(f"SERVER READY | reason={reason}")
    return CURRENT_SERVER_LOG


def _parse_chat_response(data):
    choice = data["choices"][0]
    msg = choice.get("message") or {}
    usage = data.get("usage") or {}
    details = usage.get("completion_tokens_details") or {}
    reasoning_tokens = details.get("reasoning_tokens", usage.get("reasoning_tokens"))
    return {
        "visible": str(msg.get("content") or "").strip(),
        "reasoning": str(msg.get("reasoning_content") or msg.get("reasoning") or "").strip(),
        "finish_reason": choice.get("finish_reason"),
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": usage.get("total_tokens"),
        "response_id": data.get("id"),
        "model_returned": data.get("model"),
        "logprobs": choice.get("logprobs"),
        "raw": data,
    }


def post_chat(payload, unit_key, call_context, timeout_s):
    last = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        if session_soft_stop_reached():
            return {
                "api_ok": False, "fatal": False, "soft_stop": True,
                "error_type": "SESSION_SOFT_STOP", "error_message": "Session soft-stop reached",
                "attempts": attempt - 1,
            }
        if server is None or server.poll() is not None:
            start_server(reason=f"before_{call_context}_{unit_key}_attempt_{attempt}")

        started_utc = datetime.now(timezone.utc).isoformat()
        st = time.perf_counter()
        try:
            resp = requests.post(API, json=payload, timeout=timeout_s)
            elapsed = time.perf_counter() - st
            if resp.status_code != 200:
                text = resp.text[:8000]
                fatal = resp.status_code in {400, 401, 403, 404, 413, 422}
                last = {
                    "api_ok": False, "fatal": fatal,
                    "error_type": f"HTTP_{resp.status_code}",
                    "error_message": text, "http_status": int(resp.status_code),
                    "attempts": attempt,
                }
                append_jsonl(EVENTS_JSONL, {
                    "event": last["error_type"], "unit_key": unit_key, "context": call_context,
                    "attempt": attempt, "elapsed_s": elapsed, "fatal": fatal,
                    "captured_utc": datetime.now(timezone.utc).isoformat(),
                })
                if fatal or attempt >= MAX_ATTEMPTS:
                    return last
                time.sleep(min(MAX_BACKOFF_S, BASE_BACKOFF_S * 2 ** (attempt - 1)))
                continue

            parsed = _parse_chat_response(resp.json())
            return {
                "api_ok": True, "fatal": False, "http_status": 200,
                "attempts": attempt, "latency_s": elapsed,
                "request_started_utc": started_utc,
                "response_received_utc": datetime.now(timezone.utc).isoformat(),
                **parsed,
            }
        except Exception as exc:
            elapsed = time.perf_counter() - st
            server_dead = server is None or server.poll() is not None
            last = {
                "api_ok": False, "fatal": False,
                "error_type": "LOCAL_SERVER_DIED" if server_dead else type(exc).__name__,
                "error_message": str(exc), "http_status": None, "attempts": attempt,
            }
            append_jsonl(EVENTS_JSONL, {
                "event": last["error_type"], "unit_key": unit_key, "context": call_context,
                "attempt": attempt, "elapsed_s": elapsed, "server_dead": server_dead,
                "captured_utc": datetime.now(timezone.utc).isoformat(),
            })
            if attempt >= MAX_ATTEMPTS:
                return last
            if server_dead:
                start_server(reason=f"after_crash_{call_context}_{unit_key}_{attempt}")
            else:
                time.sleep(min(MAX_BACKOFF_S, BASE_BACKOFF_S * 2 ** (attempt - 1)))
    return last


CURRENT_SERVER_LOG = start_server("initial")


[2026-09-09 07:14:46 UTC] SERVER COMMAND: /kaggle/temp/internvl35_8b_p4_runtime/llama serve -m /kaggle/input/notebooks/vernvern/notebookf785af4a9b/internvl35_8b_saved/InternVL3_5-8B-Q4_K_M.gguf --mmproj /kaggle/input/notebooks/vernvern/notebookf785af4a9b/internvl35_8b_saved/mmproj-model-f16.gguf --host 127.0.0.1 --port 8080 --alias internvl3.5-8b --gpu-layers all --split-mode layer --tensor-split 1,1 --ctx-size 16384 --parallel 1 --reasoning off --log-colors off --log-timestamps --log-prefix --log-verbosity 3
[llama:initial] 0.00.285.313 I cmn  common_param: common_params_print_info: verbosity = 3 (adjust with the `-lv N` CLI arg)
[llama:initial] 0.00.285.901 W srv  llama_server: -----------------
[llama:initial] 0.00.285.910 W srv  llama_server: CORS is set to allow all origins ('*') and no API key is set
[llama:initial] 0.00.285.911 W srv  llama_server: this can be a security risk (cross-origin attacks)
[llama:initial] 0.00.285.912 W srv  llama_server: more info: https://github.com/g

## Phase 1 — generate and freeze three facts per sample

All three knowledge calls use the same three same-subject demonstration
examples. Diversity comes only from the three fixed stochastic sampling seeds.
Successful calls are checkpointed immediately. Empty or repeated knowledge is
not silently regenerated; the raw output is preserved.


In [9]:
KNOWLEDGE_SIGNATURE_PAYLOAD = {
    "protocol": "P4_GKP_KNOWLEDGE_GENERATION_V1",
    "model": MODEL,
    "model_sha256": MODEL_SHA256,
    "mmproj_sha256": MMPROJ_SHA256,
    "llama_runtime_sha256": LLAMA_RUNTIME_SHA256,
    "dataset_revision": DATASET_REVISION,
    "selected_ids_sha256": EXPECTED_SELECTED_IDS_CANONICAL_SHA256,
    "gkp_xlsx_sha256": GKP_XLSX_SHA256,
    "gkp_demo_map_sha256": KNOWLEDGE_DEMO_MAP_SHA256,
    "knowledge_prompt_template_sha256": KNOWLEDGE_PROMPT_TEMPLATE_SHA256,
    "knowledge_temperature": KNOWLEDGE_TEMPERATURE,
    "knowledge_top_p": KNOWLEDGE_TOP_P,
    "knowledge_seeds": KNOWLEDGE_SEEDS,
    "knowledge_max_tokens": KNOWLEDGE_MAX_TOKENS,
    "knowledge_stop": "\\n",
    "n_knowledge": N_KNOWLEDGE,
}
KNOWLEDGE_SIGNATURE = canonical_json_sha256(KNOWLEDGE_SIGNATURE_PAYLOAD)


def knowledge_unit_key(sample_id, fact_index):
    return f"{sample_id}::k{fact_index + 1}"


def valid_knowledge_record(rec):
    if rec.get("api_ok") is not True:
        return False
    if rec.get("knowledge_signature") != KNOWLEDGE_SIGNATURE:
        return False
    if rec.get("sample_id") not in query_meta_by_id:
        return False
    if int(rec.get("fact_index", -1)) not in range(N_KNOWLEDGE):
        return False
    expected_prompt = canonical_parts_text(knowledge_prompt_parts(rec["sample_id"]))
    if rec.get("prompt_sha256") != hashlib.sha256(expected_prompt.encode("utf-8")).hexdigest():
        return False
    return True


knowledge_successes = [x for x in read_jsonl(KNOWLEDGE_RAW_JSONL) if valid_knowledge_record(x)]
knowledge_by_key = {}
for rec in knowledge_successes:
    key = knowledge_unit_key(rec["sample_id"], int(rec["fact_index"]))
    require(key not in knowledge_by_key, f"Duplicate knowledge success: {key}")
    knowledge_by_key[key] = rec

print("Knowledge checkpoint:", len(knowledge_by_key), "/", EXPECTED_N * N_KNOWLEDGE)

if P4_PHASE == "KNOWLEDGE":
    for sample_id in tqdm(selected_ids, desc=f"{MODEL_KIND} P4 knowledge generation"):
        if session_soft_stop_reached():
            print("SESSION SOFT STOP: save this Kaggle version and resume the same KNOWLEDGE phase.")
            break

        for fact_index in range(N_KNOWLEDGE):
            key = knowledge_unit_key(sample_id, fact_index)
            if key in knowledge_by_key:
                continue

            parts = knowledge_prompt_parts(sample_id)
            prompt_text = canonical_parts_text(parts)
            payload = knowledge_request_payload(sample_id, fact_index)
            result = post_chat(payload, key, "KNOWLEDGE", REQUEST_TIMEOUT_S)

            if result.get("soft_stop"):
                break
            if result.get("fatal"):
                raise RuntimeError(f"Fatal knowledge request failure {key}: {result.get('error_type')}: {result.get('error_message')}")
            if result.get("api_ok") is not True:
                print("Knowledge call unresolved:", key, result.get("error_type"))
                continue

            fact = str(result.get("visible") or "").strip()
            if fact.lower().startswith("knowledge:"):
                fact = fact.split(":", 1)[1].strip()

            rec = {
                "knowledge_signature": KNOWLEDGE_SIGNATURE,
                "sample_id": sample_id,
                "execution_order": query_meta_by_id[sample_id]["execution_order"],
                "subject": query_meta_by_id[sample_id]["subject"],
                "fact_index": fact_index,
                "fact_number": fact_index + 1,
                "seed": KNOWLEDGE_SEEDS[fact_index],
                "temperature": KNOWLEDGE_TEMPERATURE,
                "top_p": KNOWLEDGE_TOP_P,
                "max_tokens": KNOWLEDGE_MAX_TOKENS,
                "demo_ids": KNOWLEDGE_DEMO_MAP[query_meta_by_id[sample_id]["subject"]],
                "prompt_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
                "generated_knowledge": fact,
                "knowledge_nonempty": bool(fact),
                "reasoning_content": result.get("reasoning", ""),
                "finish_reason": result.get("finish_reason"),
                "prompt_tokens": result.get("prompt_tokens"),
                "completion_tokens": result.get("completion_tokens"),
                "reasoning_tokens": result.get("reasoning_tokens"),
                "total_tokens": result.get("total_tokens"),
                "latency_s": result.get("latency_s"),
                "attempts": result.get("attempts"),
                "api_ok": True,
                "http_status": 200,
            }
            append_jsonl(KNOWLEDGE_RAW_JSONL, rec)
            knowledge_by_key[key] = rec

        if session_soft_stop_reached():
            break

    knowledge_successes = list(knowledge_by_key.values())

# Build compact bank only when 3 successful calls exist for every sample.
complete_bank = []
for sid in selected_ids:
    recs = [knowledge_by_key.get(knowledge_unit_key(sid, i)) for i in range(N_KNOWLEDGE)]
    if all(r is not None for r in recs):
        complete_bank.append({
            "sample_id": sid,
            "execution_order": query_meta_by_id[sid]["execution_order"],
            "subject": query_meta_by_id[sid]["subject"],
            "facts": [r["generated_knowledge"] for r in recs],
            "fact_seeds": [r["seed"] for r in recs],
            "demo_ids": KNOWLEDGE_DEMO_MAP[query_meta_by_id[sid]["subject"]],
            "knowledge_signature": KNOWLEDGE_SIGNATURE,
        })

if len(complete_bank) == EXPECTED_N:
    atomic_text(
        KNOWLEDGE_BANK_JSONL,
        "".join(json.dumps(x, ensure_ascii=False) + "\n" for x in complete_bank)
    )
    atomic_json(KNOWLEDGE_MANIFEST_JSON, {
        "status": "COMPLETE",
        "rows": len(complete_bank),
        "knowledge_signature": KNOWLEDGE_SIGNATURE,
        "signature_payload": KNOWLEDGE_SIGNATURE_PAYLOAD,
        "knowledge_bank_sha256": sha_file(KNOWLEDGE_BANK_JSONL),
        "generated_utc": datetime.now(timezone.utc).isoformat(),
    })
    print("KNOWLEDGE PHASE COMPLETE: 300 samples × 3 facts.")
    print("Save Version. For the next Kaggle session attach this output and set P4_PHASE='INFERENCE'.")
else:
    print(f"KNOWLEDGE PHASE PARTIAL: {len(complete_bank)}/300 samples have all three facts.")
    if P4_PHASE == "INFERENCE":
        print("Inference phase will now search attached Kaggle Inputs for a complete knowledge bank.")


Knowledge checkpoint: 0 / 900
KNOWLEDGE PHASE PARTIAL: 0/300 samples have all three facts.
Inference phase will now search attached Kaggle Inputs for a complete knowledge bank.


## Phase 2 — load the frozen knowledge bank and verify logprob capability

Set `P4_PHASE = "INFERENCE"` only after the previous phase produced a complete
`p4_knowledge_bank.jsonl` + `p4_knowledge_manifest.json`, then attach that
Kaggle output to the fresh session.


In [10]:
def discover_complete_knowledge_bank():
    candidates = discover_named(KNOWLEDGE_BANK_JSONL.name)
    accepted = []
    diag = []
    for bank in candidates:
        manifest = bank.parent / KNOWLEDGE_MANIFEST_JSON.name
        try:
            if not manifest.is_file():
                continue
            m = json.loads(manifest.read_text(encoding="utf-8"))
            rows = read_jsonl(bank)
            ok = (
                m.get("status") == "COMPLETE"
                and m.get("knowledge_signature") == KNOWLEDGE_SIGNATURE
                and len(rows) == EXPECTED_N
                and m.get("knowledge_bank_sha256") == sha_file(bank)
                and all(x.get("knowledge_signature") == KNOWLEDGE_SIGNATURE for x in rows)
            )
            diag.append({"bank": str(bank), "manifest": str(manifest), "accepted": ok})
            if ok:
                accepted.append((bank, manifest, rows))
        except Exception as exc:
            diag.append({"bank": str(bank), "error": repr(exc)})
    require(accepted, "No complete exact-signature P4 knowledge bank found. Diagnostics:\n" + json.dumps(diag, indent=2))
    return sorted(accepted, key=lambda x: str(x[0]))[0]


if P4_PHASE == "INFERENCE":
    KNOWLEDGE_BANK_PATH, KNOWLEDGE_MANIFEST_PATH, knowledge_bank_rows = discover_complete_knowledge_bank()
    bank_by_id = {x["sample_id"]: x for x in knowledge_bank_rows}
    require(set(bank_by_id) == set(selected_ids), "Knowledge bank ID coverage drift")
    require(all(len(bank_by_id[sid]["facts"]) == 3 for sid in selected_ids), "Knowledge bank does not contain exactly 3 facts/sample")
    print("Using frozen knowledge bank:", KNOWLEDGE_BANK_PATH)
else:
    bank_by_id = {x["sample_id"]: x for x in complete_bank} if len(complete_bank) == EXPECTED_N else {}


FINAL_ANSWER_RE = re.compile(r"(?:^|\n)Answer:\s*([A-Z])\s*\Z")


def _parse_visible_reasoning_and_answer(visible_response, letters):
    text = str(visible_response or "").rstrip()
    match = FINAL_ANSWER_RE.search(text)
    require(match is not None, "Response does not end with exact final line Answer: $LETTER")
    label = match.group(1)
    require(label in letters, f"Illegal final answer label: {label!r}")
    line_start = match.start()
    if line_start < len(text) and text[line_start:line_start + 1] == "\n":
        line_start += 1
    return {
        "parsed_answer": label,
        "visible_reasoning": text[:match.start()].strip(),
        "final_answer_line": text[line_start:].strip(),
    }


def _label_from_candidate_token(token, letters):
    match = re.fullmatch(r"\s*([A-Z])\s*", str(token or ""))
    return match.group(1) if match and match.group(1) in letters else None


def _candidate_logprobs_at_answer_position(logprobs_obj, visible_response, letters):
    require(isinstance(logprobs_obj, dict), "Missing logprobs object")
    content = logprobs_obj.get("content")
    require(isinstance(content, list) and content, "Missing logprobs.content")

    parsed = _parse_visible_reasoning_and_answer(visible_response, letters)
    token_strings = [str(item.get("token") or "") for item in content]
    token_text = "".join(token_strings).rstrip()
    match = FINAL_ANSWER_RE.search(token_text)
    require(match is not None, "Token stream has no exact final Answer: $LETTER line")
    require(match.group(1) == parsed["parsed_answer"],
            f"Visible/token answer mismatch: {parsed['parsed_answer']} vs {match.group(1)}")

    answer_char_index = int(match.start(1))
    answer_token_index = None
    offset = 0
    for idx, token in enumerate(token_strings):
        next_offset = offset + len(token)
        if offset <= answer_char_index < next_offset:
            answer_token_index = idx
            break
        offset = next_offset
    require(answer_token_index is not None, "Could not align answer letter to token stream")

    answer_item = content[answer_token_index]
    answer_token = str(answer_item.get("token") or "")
    require(_label_from_candidate_token(answer_token, letters) == parsed["parsed_answer"],
            f"Answer letter is not isolated in a scoreable label token: {answer_token!r}")

    top = [dict(x) for x in (answer_item.get("top_logprobs") or [])]
    top.append({"token": answer_item.get("token"),
                "logprob": answer_item.get("logprob"),
                "_generated_token": True})

    by_label = defaultdict(list)
    for item in top:
        label = _label_from_candidate_token(item.get("token"), letters)
        lp = item.get("logprob")
        if label is not None and lp is not None:
            by_label[label].append(float(lp))

    option_logprobs = {
        letter: (logsumexp(by_label[letter]) if by_label.get(letter) else None)
        for letter in letters
    }
    option_raw_probabilities = {
        letter: (math.exp(lp) if lp is not None else None)
        for letter, lp in option_logprobs.items()
    }
    coverage_complete = all(option_raw_probabilities[x] is not None for x in letters)

    option_normalized_probabilities = {x: None for x in letters}
    if coverage_complete:
        z = sum(option_raw_probabilities.values())
        if z > 0:
            option_normalized_probabilities = {x: option_raw_probabilities[x] / z for x in letters}

    answer_lp = answer_item.get("logprob")
    answer_probability = math.exp(float(answer_lp)) if answer_lp is not None else None

    generated_token_logprobs = [
        {"token_index": idx, "token": item.get("token"), "logprob": item.get("logprob")}
        for idx, item in enumerate(content)
    ]

    return {
        **parsed,
        "generated_label": parsed["parsed_answer"],
        "generated_token": answer_token,
        "generated_logprob": answer_lp,
        "generated_probability": answer_probability,
        "answer_token_index": int(answer_token_index),
        "answer_char_index_in_token_stream": int(answer_char_index),
        "option_logprobs": option_logprobs,
        "option_raw_probabilities": option_raw_probabilities,
        "option_normalized_probabilities": option_normalized_probabilities,
        "option_probability_coverage_complete": bool(coverage_complete),
        "top_logprobs_at_answer": top,
        "generated_token_logprobs": generated_token_logprobs,
    }

def logprob_capability_preflight():
    if P4_PHASE != "INFERENCE":
        return
    payload = {
        "model": MODEL_API_NAME,
        "messages": [{"role": "user", "content": "Reply with exactly A and nothing else."}],
        "temperature": 0.0,
        "seed": BASELINE_SEED,
        "max_tokens": 4,
        "stream": False,
        "logprobs": True,
        "top_logprobs": 5,
    }

    result = post_chat(payload, "logprob_preflight", "LOGPROB_PREFLIGHT", REQUEST_TIMEOUT_S)
    require(result.get("api_ok") is True, f"Logprob preflight failed: {result}")
    require(isinstance(result.get("logprobs"), dict), "llama.cpp chat endpoint returned no logprobs")
    require(result["logprobs"].get("content"), "llama.cpp returned empty logprobs.content")
    print("LOGPROB PREFLIGHT: PASS")
    print("This runtime exposes OpenAI-compatible generated-token logprobs.")


logprob_capability_preflight()


Using frozen knowledge bank: /kaggle/input/notebooks/vernvern/internvl3-5-8b-knowledge/internvl_p4_gkp_3facts_logprob/p4_knowledge_bank.jsonl
[llama:initial] 0.04.154.215 I slot get_availabl: id  0 | task -1 | selected slot by LRU, t_last = -1
[llama:initial] 0.04.154.296 I slot launch_slot_: id  0 | task 0 | processing task, is_child = 0
[llama:initial] 0.04.355.406 I slot print_timing: id  0 | task 0 | prompt eval time =      89.94 ms /    16 tokens (    5.62 ms per token,   177.89 tokens per second)
[llama:initial] 0.04.355.416 I slot print_timing: id  0 | task 0 |        eval time =     109.94 ms /     4 tokens (   36.65 ms per token,    27.29 tokens per second)
[llama:initial] 0.04.355.418 I slot print_timing: id  0 | task 0 |       total time =     199.89 ms /    20 tokens
LOGPROB PREFLIGHT: PASS
This runtime exposes OpenAI-compatible generated-token logprobs.
[llama:initial] 0.04.355.418 I slot print_timing: id  0 | task 0 |    graphs reused =          3
[llama:initial] 0.04.355

## Zero-shot knowledge integration — three calls per sample

No demonstrations are sent in Round 2. Each call uses exactly one frozen
generated fact and the exact prompt below:

```text
[TARGET IMAGE(S)]

Knowledge:
{GENERATED_KNOWLEDGE}

Question text

A. option 1
B. option 2
C. option 3
...
[last available letter]. option N

Answer the preceding multiple choice question.
The last line of your response should be of the following format:
'Answer: $LETTER' (without quotes) where LETTER is one of options.
Explain your reasoning before answering.
```

The notebook stores the visible reasoning, the parsed final answer, the option
logprobs/probabilities at the final answer-letter token, the top logprobs at
that position, and each generated token's own logprob.


In [11]:
INFERENCE_SIGNATURE_PAYLOAD = {
    "protocol": "P4_GKP_KNOWLEDGE_INTEGRATION_REASONING_FINALANSWER_LOGPROB_V2",
    "model": MODEL,
    "model_sha256": MODEL_SHA256,
    "mmproj_sha256": MMPROJ_SHA256,
    "dataset_revision": DATASET_REVISION,
    "selected_ids_sha256": EXPECTED_SELECTED_IDS_CANONICAL_SHA256,
    "knowledge_signature": KNOWLEDGE_SIGNATURE,
    "inference_prompt_template_sha256": INFERENCE_PROMPT_TEMPLATE_SHA256,
    "temperature": 0.0,
    "seed": BASELINE_SEED,
    "max_tokens": BASELINE_MAX_TOKENS,
    "top_p": None,
    "top_k": None,
    "logprobs": True,
    "top_logprobs": LOGPROBS_TOP_N,
    "response_format": "visible reasoning followed by exact final line Answer: $LETTER",
    "logprob_position": "final answer-letter token after visible reasoning",
    "aggregation": "argmax_a max_m p(a | knowledge_m + question)",
}
INFERENCE_SIGNATURE = canonical_json_sha256(INFERENCE_SIGNATURE_PAYLOAD)


def inference_unit_key(sample_id, fact_index):
    return f"{sample_id}::f{fact_index + 1}"


def valid_inference_record(rec):
    if rec.get("api_ok") is not True:
        return False
    if rec.get("inference_signature") != INFERENCE_SIGNATURE:
        return False
    if rec.get("sample_id") not in query_meta_by_id:
        return False
    if int(rec.get("fact_index", -1)) not in range(N_KNOWLEDGE):
        return False
    return True


inference_successes = [x for x in read_jsonl(INFERENCE_RAW_JSONL) if valid_inference_record(x)]
inference_by_key = {}
for rec in inference_successes:
    key = inference_unit_key(rec["sample_id"], int(rec["fact_index"]))
    require(key not in inference_by_key, f"Duplicate inference success: {key}")
    inference_by_key[key] = rec

print("Inference checkpoint:", len(inference_by_key), "/", EXPECTED_N * N_KNOWLEDGE)

if P4_PHASE == "INFERENCE":
    for sid in tqdm(selected_ids, desc=f"{MODEL_KIND} P4 knowledge integration"):
        if session_soft_stop_reached():
            print("SESSION SOFT STOP: save this Kaggle version and resume the INFERENCE phase.")
            break

        facts = bank_by_id[sid]["facts"]
        for fact_index, fact in enumerate(facts):
            key = inference_unit_key(sid, fact_index)
            if key in inference_by_key:
                continue

            payload = inference_request_payload(sid, fact)
            prompt_text = canonical_parts_text(inference_prompt_parts(sid, fact))
            result = post_chat(payload, key, "INFERENCE", REQUEST_TIMEOUT_S)

            if result.get("soft_stop"):
                break
            if result.get("fatal"):
                raise RuntimeError(f"Fatal inference failure {key}: {result.get('error_type')}: {result.get('error_message')}")
            if result.get("api_ok") is not True:
                print("Inference unresolved:", key, result.get("error_type"))
                continue

            qmeta = query_meta_by_id[sid]
            try:
                score = _candidate_logprobs_at_answer_position(
                    result.get("logprobs"), result.get("visible", ""), qmeta["choice_letters"]
                )
                scoring_ok = score["generated_label"] is not None
                scoring_error = None
            except Exception as exc:
                score = {
                    "parsed_answer": None,
                    "visible_reasoning": None,
                    "final_answer_line": None,
                    "generated_label": None,
                    "generated_token": None,
                    "generated_logprob": None,
                    "generated_probability": None,
                    "answer_token_index": None,
                    "answer_char_index_in_token_stream": None,
                    "option_logprobs": {x: None for x in qmeta["choice_letters"]},
                    "option_raw_probabilities": {x: None for x in qmeta["choice_letters"]},
                    "option_normalized_probabilities": {x: None for x in qmeta["choice_letters"]},
                    "option_probability_coverage_complete": False,
                    "top_logprobs_at_answer": [],
                    "generated_token_logprobs": [],
                }
                scoring_ok = False
                scoring_error = repr(exc)

            rec = {
                "inference_signature": INFERENCE_SIGNATURE,
                "knowledge_signature": KNOWLEDGE_SIGNATURE,
                "sample_id": sid,
                "execution_order": qmeta["execution_order"],
                "subject": qmeta["subject"],
                "fact_index": fact_index,
                "fact_number": fact_index + 1,
                "knowledge": fact,
                "prompt_sha256": hashlib.sha256(prompt_text.encode("utf-8")).hexdigest(),
                "temperature": 0.0,
                "seed": BASELINE_SEED,
                "max_tokens": BASELINE_MAX_TOKENS,
                "logprobs_requested": True,
                "top_logprobs_requested": LOGPROBS_TOP_N,
                "visible_response": result.get("visible", ""),
                "native_reasoning_content": result.get("reasoning", ""),
                "finish_reason": result.get("finish_reason"),
                "prompt_tokens": result.get("prompt_tokens"),
                "completion_tokens": result.get("completion_tokens"),
                "reasoning_tokens": result.get("reasoning_tokens"),
                "total_tokens": result.get("total_tokens"),
                "latency_s": result.get("latency_s"),
                "attempts": result.get("attempts"),
                "scoring_ok": scoring_ok,
                "scoring_error": scoring_error,
                **score,
                "api_ok": True,
                "http_status": 200,
            }
            append_jsonl(INFERENCE_RAW_JSONL, rec)
            inference_by_key[key] = rec

        if session_soft_stop_reached():
            break

print("Inference successes now:", len(inference_by_key), "/", EXPECTED_N * N_KNOWLEDGE)


Inference checkpoint: 0 / 900


internvl P4 knowledge integration:   0%|          | 0/300 [00:00<?, ?it/s]

[llama:initial] 0.04.578.874 I slot get_availabl: id  0 | task -1 | selected slot by LRU, t_last = 499215522
[llama:initial] 0.04.581.962 I slot launch_slot_: id  0 | task 5 | processing task, is_child = 0
[llama:initial] 0.09.144.859 I slot print_timing: id  0 | task 5 | n_gen =    124, tg =  40.94 t/s, tg_3s =  41.25 t/s
[llama:initial] 0.12.160.693 I slot print_timing: id  0 | task 5 | n_gen =    247, tg =  40.86 t/s, tg_3s =  40.78 t/s
[llama:initial] 0.15.165.211 I slot print_timing: id  0 | task 5 | n_gen =    369, tg =  40.78 t/s, tg_3s =  40.61 t/s
[llama:initial] 0.16.933.006 I slot print_timing: id  0 | task 5 | prompt eval time =    1557.17 ms /  1191 tokens (    1.31 ms per token,   764.85 tokens per second)
[llama:initial] 0.16.933.016 I slot print_timing: id  0 | task 5 |        eval time =   10792.61 ms /   440 tokens (   24.58 ms per token,    40.68 tokens per second)
[llama:initial] 0.16.933.018 I slot print_timing: id  0 | task 5 |       total time =   12349.78 ms /  

## Aggregate the three knowledge-conditioned calls with the GKP max-probability rule

In [12]:
# ------------------------------------------------------------------
# Call-level output: reasoning + final answer + answer-position logprobs.
# ------------------------------------------------------------------
call_rows = []
for sid in selected_ids:
    qmeta = query_meta_by_id[sid]
    for fact_index in range(N_KNOWLEDGE):
        call = inference_by_key.get(inference_unit_key(sid, fact_index))
        if call is None:
            continue
        call_rows.append({
            "sample_id": sid,
            "execution_order": qmeta["execution_order"],
            "subject": qmeta["subject"],
            "fact_index": fact_index,
            "fact_number": fact_index + 1,
            "knowledge": call.get("knowledge"),
            "visible_reasoning": call.get("visible_reasoning"),
            "parsed_answer": call.get("parsed_answer"),
            "final_answer_line": call.get("final_answer_line"),
            "visible_response": call.get("visible_response"),
            "native_reasoning_content": call.get("native_reasoning_content", ""),
            "answer_token_index": call.get("answer_token_index"),
            "answer_token": call.get("generated_token"),
            "answer_token_logprob": call.get("generated_logprob"),
            "answer_token_probability": call.get("generated_probability"),
            "option_logprobs_json": json.dumps(call.get("option_logprobs"), ensure_ascii=False),
            "option_raw_probabilities_json": json.dumps(call.get("option_raw_probabilities"), ensure_ascii=False),
            "option_normalized_probabilities_json": json.dumps(call.get("option_normalized_probabilities"), ensure_ascii=False),
            "option_probability_coverage_complete": call.get("option_probability_coverage_complete"),
            "top_logprobs_at_answer_json": json.dumps(call.get("top_logprobs_at_answer"), ensure_ascii=False),
            "generated_token_logprobs_json": json.dumps(call.get("generated_token_logprobs"), ensure_ascii=False),
            "scoring_ok": call.get("scoring_ok"),
            "scoring_error": call.get("scoring_error"),
            "prompt_tokens": call.get("prompt_tokens"),
            "completion_tokens": call.get("completion_tokens"),
            "reasoning_tokens": call.get("reasoning_tokens"),
            "total_tokens": call.get("total_tokens"),
            "latency_s": call.get("latency_s"),
        })

inference_calls_df = pd.DataFrame(call_rows)
if len(inference_calls_df):
    inference_calls_df = inference_calls_df.sort_values(["execution_order", "fact_index"]).reset_index(drop=True)
inference_calls_df.to_csv(INFERENCE_CALLS_CSV, index=False)
print("Call-level reasoning/answer/logprob CSV:", INFERENCE_CALLS_CSV, "rows=", len(inference_calls_df))


# ------------------------------------------------------------------
# GKP aggregation:
#   final(a) = max_m p(a | knowledge_m + question)
#   prediction = argmax_a final(a)
# ------------------------------------------------------------------
aggregate_rows = []

for sid in selected_ids:
    qmeta = query_meta_by_id[sid]
    calls = [inference_by_key.get(inference_unit_key(sid, i)) for i in range(N_KNOWLEDGE)]
    all_three_present = all(x is not None for x in calls)

    option_max_raw = {letter: None for letter in qmeta["choice_letters"]}
    winning_fact_for_option = {letter: None for letter in qmeta["choice_letters"]}

    if all_three_present:
        for letter in qmeta["choice_letters"]:
            candidates = []
            for call in calls:
                p = call["option_raw_probabilities"].get(letter)
                if p is not None:
                    candidates.append((float(p), int(call["fact_index"])))
            if candidates:
                p, fact_index = max(candidates, key=lambda x: x[0])
                option_max_raw[letter] = p
                winning_fact_for_option[letter] = fact_index + 1

    exact_probability_coverage = (
        all_three_present
        and all(call.get("option_probability_coverage_complete") for call in calls)
        and all(option_max_raw[x] is not None for x in qmeta["choice_letters"])
    )

    final_prediction = None
    aggregation_method = None
    fallback_prediction = None

    if exact_probability_coverage:
        final_prediction = max(
            qmeta["choice_letters"],
            key=lambda letter: (option_max_raw[letter], -qmeta["choice_letters"].index(letter)),
        )
        aggregation_method = "exact_argmax_a_max_m_raw_option_probability"
    elif all_three_present:
        # Diagnostic-only fallback: each greedy call's generated label probability.
        # It is NOT used for headline accuracy unless exact option coverage is complete.
        usable = [
            (call.get("generated_probability"), call.get("generated_label"), int(call["fact_index"]))
            for call in calls
            if call.get("generated_probability") is not None and call.get("generated_label") in qmeta["choice_letters"]
        ]
        if usable:
            best = max(usable, key=lambda x: x[0])
            fallback_prediction = best[1]

    aggregate_rows.append({
        "sample_id": sid,
        "execution_order": qmeta["execution_order"],
        "subject": qmeta["subject"],
        "topic_difficulty": qmeta["topic_difficulty"],
        "img_type": qmeta["img_type"],
        "gold_answer": qmeta["gold_answer"],
        "facts_json": json.dumps(bank_by_id[sid]["facts"], ensure_ascii=False) if sid in bank_by_id else None,
        "call_visible_reasoning_json": json.dumps([x.get("visible_reasoning") if x else None for x in calls], ensure_ascii=False),
        "call_parsed_answers_json": json.dumps([x.get("parsed_answer") if x else None for x in calls]),
        "call_final_answer_lines_json": json.dumps([x.get("final_answer_line") if x else None for x in calls], ensure_ascii=False),
        "call_generated_labels_json": json.dumps([x.get("generated_label") if x else None for x in calls]),
        "call_generated_probabilities_json": json.dumps([x.get("generated_probability") if x else None for x in calls]),
        "call_option_logprobs_json": json.dumps([x.get("option_logprobs") if x else None for x in calls], ensure_ascii=False),
        "call_option_raw_probabilities_json": json.dumps([x.get("option_raw_probabilities") if x else None for x in calls], ensure_ascii=False),
        "option_max_raw_probabilities_json": json.dumps(option_max_raw),
        "winning_fact_for_option_json": json.dumps(winning_fact_for_option),
        "all_three_inference_calls_present": all_three_present,
        "exact_probability_coverage": exact_probability_coverage,
        "final_prediction": final_prediction,
        "fallback_prediction_diagnostic_only": fallback_prediction,
        "aggregation_method": aggregation_method,
        "strict_correct": bool(final_prediction == qmeta["gold_answer"]) if final_prediction is not None else False,
    })

aggregate = pd.DataFrame(aggregate_rows).sort_values("execution_order").reset_index(drop=True)
aggregate.to_csv(FINAL_RESULTS_CSV, index=False)

headline = aggregate[aggregate["exact_probability_coverage"] == True].copy()
coverage = len(headline) / EXPECTED_N

summary = {
    "protocol": "P4-GKP-3Facts-LogProb-Max",
    "model": MODEL,
    "phase": P4_PHASE,
    "n_eval": EXPECTED_N,
    "exact_logprob_coverage_n": int(len(headline)),
    "exact_logprob_coverage": float(coverage),
    "strict_accuracy_on_exact_coverage": (
        float(headline["strict_correct"].mean()) if len(headline) else None
    ),
    "knowledge_signature": KNOWLEDGE_SIGNATURE,
    "inference_signature": INFERENCE_SIGNATURE,
    "knowledge_prompt_template_sha256": KNOWLEDGE_PROMPT_TEMPLATE_SHA256,
    "inference_prompt_template_sha256": INFERENCE_PROMPT_TEMPLATE_SHA256,
    "aggregation": "argmax_a max_m p(a | knowledge_m + question)",
    "logprobs_top_n": LOGPROBS_TOP_N,
    "knowledge_generation": {
        "n_facts": N_KNOWLEDGE,
        "same_subject_demo_count": KNOWLEDGE_DEMOS_PER_SUBJECT,
        "demo_selection": "first three spreadsheet rows of exact subject",
        "temperature": KNOWLEDGE_TEMPERATURE,
        "top_p": KNOWLEDGE_TOP_P,
        "seeds": KNOWLEDGE_SEEDS,
        "max_tokens": KNOWLEDGE_MAX_TOKENS,
        "stop": "\\n",
    },
    "inference_generation": {
        "temperature": 0.0,
        "seed": BASELINE_SEED,
        "max_tokens": BASELINE_MAX_TOKENS,
        "top_p": None,
        "top_k": None,
        "demonstrations_sent": False,
        "visible_reasoning_requested": True,
        "required_final_line": "Answer: $LETTER",
        "logprobs": True,
        "logprob_position": "final answer-letter token after visible reasoning",
    },
}
atomic_json(SUMMARY_JSON, summary)

print(json.dumps(summary, indent=2))
display(aggregate.head(10))

if P4_PHASE == "INFERENCE":
    if len(headline) == EXPECTED_N:
        print("P4 FINAL: COMPLETE 300/300 exact option-logprob coverage.")
    else:
        print(
            f"P4 PARTIAL/DIAGNOSTIC: {len(headline)}/300 samples have complete "
            "A..N option coverage in top_logprobs. Missing probabilities were NOT fabricated."
        )


Call-level reasoning/answer/logprob CSV: /kaggle/working/internvl_p4_gkp_3facts_logprob/p4_inference_calls_reasoning_logprobs.csv rows= 900
{
  "protocol": "P4-GKP-3Facts-LogProb-Max",
  "model": "InternVL3.5-8B-Q4_K_M",
  "phase": "INFERENCE",
  "n_eval": 300,
  "exact_logprob_coverage_n": 276,
  "exact_logprob_coverage": 0.92,
  "strict_accuracy_on_exact_coverage": 0.4891304347826087,
  "knowledge_signature": "9f76e37f046de9525f1c3f4209c5b8ae5ed427fefeaccfc7ac5bb0ac64288635",
  "inference_signature": "417e4bd49e12ee2920d394a57555d4d9b25e631c1f61877a6dde3a4e1fac7dbb",
  "knowledge_prompt_template_sha256": "2bf246fa3aad7cfba1afac080599455e0e9c01525c250d65c47062fe9484c99b",
  "inference_prompt_template_sha256": "4523f006dffca5b877b78cce49880e1a33b4e936a0ee73d1bdfa3a1b36c92f1d",
  "aggregation": "argmax_a max_m p(a | knowledge_m + question)",
  "logprobs_top_n": 20,
  "knowledge_generation": {
    "n_facts": 3,
    "same_subject_demo_count": 3,
    "demo_selection": "first three spreadsh

,sample_id,execution_order,subject,topic_difficulty,img_type,gold_answer,facts_json,call_visible_reasoning_json,call_parsed_answers_json,call_final_answer_lines_json,...,call_option_logprobs_json,call_option_raw_probabilities_json,option_max_raw_probabilities_json,winning_fact_for_option_json,all_three_inference_calls_present,exact_probability_coverage,final_prediction,fallback_prediction_diagnostic_only,aggregation_method,strict_correct
0,test_Public_Health_189,0,Public_Health,Hard,['Tables'],G,"[""In a serial test, both screening tests must ...","[""To determine the changes in sensitivity and ...","[""E"", ""G"", ""G""]","[""Answer: E"", ""Answer: G"", ""Answer: G""]",...,"[{""A"": -23.61551284790039, ""B"": -20.4827938079...","[{""A"": 5.545147245170211e-11, ""B"": 1.271849358...","{""A"": 1.8168416221838133e-05, ""B"": 3.151391567...","{""A"": 2, ""B"": 2, ""C"": 2, ""D"": 2, ""E"": 1, ""F"": ...",True,True,E,None,exact_argmax_a_max_m_raw_option_probability,False
1,validation_Manage_30,1,Manage,Medium,['Tables'],G,"[""### Knowledge about the Concepts in the Inpu...","[""To determine which year the least-squares re...","[""B"", ""I"", ""B""]","[""Answer: B"", ""Answer: I"", ""Answer: B""]",...,"[{""A"": -21.337783813476562, ""B"": 0.69314372348...","[{""A"": 5.409015597277523e-10, ""B"": 1.999993085...","{""A"": 6.164013620903031e-10, ""B"": 1.9999930858...","{""A"": 3, ""B"": 1, ""C"": 1, ""D"": 1, ""E"": 1, ""F"": ...",True,True,I,None,exact_argmax_a_max_m_raw_option_probability,False
2,test_Psychology_96,2,Psychology,Medium,"['Plots and Charts', 'Photographs']",B,"[""A prosopagnosic might use top-down processin...","[""To determine the most accurate top-down proc...","[""B"", ""B"", ""B""]","[""Answer: B"", ""Answer: B"", ""Answer: B""]",...,"[{""A"": -23.404666900634766, ""B"": 0.69314718055...","[{""A"": 6.846716007124819e-11, ""B"": 2.0, ""C"": 8...","{""A"": 3.651068205065633e-10, ""B"": 2.0, ""C"": 6....","{""A"": 2, ""B"": 1, ""C"": 2, ""D"": 2, ""E"": 3, ""F"": ...",True,True,B,None,exact_argmax_a_max_m_raw_option_probability,True
3,test_Math_164,3,Math,Hard,['Diagrams'],B,"[""To solve the problem of minimizing the poten...","[""To solve the problem of minimizing the poten...","[""B"", ""B"", ""B""]","[""Answer: B"", ""Answer: B"", ""Answer: B""]",...,"[{""A"": -13.486889839172363, ""B"": 0.69314265059...","[{""A"": 1.389050914755504e-06, ""B"": 1.999990940...","{""A"": 9.682717883719795e-06, ""B"": 1.9999914169...","{""A"": 3, ""B"": 2, ""C"": 3, ""D"": 3, ""E"": 3, ""F"": ...",True,True,B,None,exact_argmax_a_max_m_raw_option_probability,True
4,validation_Finance_13,4,Finance,Easy,['Tables'],H,"[""To determine the average price per share of ...","[""To determine the average price per share of ...","[""H"", ""B"", ""C""]","[""Answer: H"", ""Answer: B"", ""Answer: C""]",...,"[{""A"": -9.331663131713867, ""B"": -3.99563431739...","[{""A"": 8.857480296503167e-05, ""B"": 0.018395773...","{""A"": 0.0018626684964863857, ""B"": 0.9338626051...","{""A"": 3, ""B"": 2, ""C"": 3, ""D"": 3, ""E"": 3, ""F"": ...",True,True,H,None,exact_argmax_a_max_m_raw_option_probability,True
5,test_Public_Health_243,5,Public_Health,Medium,['Diagrams'],F,"[""The Infection Window Period is typically det...","[""To determine the first diagnostic test or si...","[""A"", ""A"", ""A""]","[""Answer: A"", ""Answer: A"", ""Answer: A""]",...,"[{""A"": 0.6931471805599453, ""B"": -21.6686630249...","[{""A"": 2.0, ""B"": 3.885252227121239e-10, ""C"": 6...","{""A"": 2.0, ""B"": 2.6500236276762348e-09, ""C"": 5...","{""A"": 1, ""B"": 2, ""C"": 2, ""D"": 2, ""E"": 2, ""F"": ...",True,True,A,None,exact_argmax_a_max_m_raw_option_probability,False
6,test_Electronics_245,6,Electronics,Hard,['Diagrams'],D,"[""To find \\( v_2(t) \\) using Laplace transfo...","[""To find \\( v_2(t) \\) using Laplace transfo...","[""D"", ""D"", ""D""]","[""Answer: D"", ""Answer: D"", ""Answer: D""]",...,"[{""A"": -8.327218055725098, ""B"": -11.0921487808...","[{""A"": 0.0002418439

P4 PARTIAL/DIAGNOSTIC: 276/300 samples have complete A..N option coverage in top_logprobs. Missing probabilities were NOT fabricated.


## Paper/thesis-ready run description and canonical artifacts

In [13]:
readme = f"""# {EXPERIMENT_ID}

## Protocol
P4 implements multimodal Generated Knowledge Prompting (GKP) with the same model
used as both knowledge generator and inference model.

### Phase 1 — knowledge generation
- Exactly 3 knowledge calls per Eval300 sample.
- Every knowledge-generation prompt contains the first 3 spreadsheet
  demonstrations from the exact same subject.
- Each demonstration supplies its image(s), question, and reference `knowledge` field from `{GKP_XLSX_NAME}`.
- The target knowledge-generation input supplies only its image(s) and question; answer choices and gold answers are not sent in this phase.
- Knowledge calls use temperature `{KNOWLEDGE_TEMPERATURE}`, nucleus
  `top_p={KNOWLEDGE_TOP_P}`, seeds `{KNOWLEDGE_SEEDS}`, max 64 tokens, and stop
  at newline.
- Generated facts are checkpointed before any answer-inference phase.

### Phase 2 — zero-shot knowledge integration
- No demonstration is sent to the model.
- Each of the 3 frozen facts is evaluated in one separate call.
- Temperature is 0; seed is `{BASELINE_SEED}`; max output cap is
  `{BASELINE_MAX_TOKENS}`; top-p/top-k are not explicitly sent.
- OpenAI-compatible `logprobs=true`, `top_logprobs={LOGPROBS_TOP_N}` are requested.
- The prompt requests visible reasoning and requires the last line to be exactly `Answer: $LETTER`.
- Visible reasoning, parsed answer, and answer-position logprobs are stored.
- For each answer option `a`, the aggregate score is the maximum raw answer-token
  probability over the 3 knowledge-augmented calls.
- Final prediction is `argmax_a max_m p(a | q, k_m)`.
- If the server's top-logprob list does not contain every legal option label, the
  missing probabilities are not invented; the sample is excluded from headline
  exact-logprob coverage and a greedy-label fallback is retained only as a diagnostic.

## Session workflow
1. Run with `P4_PHASE = "KNOWLEDGE"`.
2. Save the Kaggle Version/output.
3. In a fresh GPU session, attach that output and this same input bundle.
4. Set `P4_PHASE = "INFERENCE"` and Run All.
5. Resume is unit-level and exact-signature checked.

## Frozen sources
- MMMU-Pro Eval300 revision: `{DATASET_REVISION}`
- MMMU demonstration revision: `{DEMO_REVISION}`
- GKP spreadsheet SHA-256: `{GKP_XLSX_SHA256}`
- GKP same-subject first-three demo map SHA-256: `{KNOWLEDGE_DEMO_MAP_SHA256}`

## Runtime
- Model: `{MODEL}`
- Model SHA-256: `{MODEL_SHA256}`
- Projector SHA-256: `{MMPROJ_SHA256}`
- llama runtime SHA-256: `{LLAMA_RUNTIME_SHA256}`
- Context: `{CTX_SIZE}`
- GPU policy: T4 ×2
- The input llama binary is never executed directly from `/kaggle/input`; a
  byte-identical SHA-verified copy under `/kaggle/temp` is used.
"""
atomic_text(README_PATH, readme)
print(readme)


# MMMUPro-Eval300__InternVL3.5-8B-Q4_K_M__P4-GKP-3Facts-LogProb-Max

## Protocol
P4 implements multimodal Generated Knowledge Prompting (GKP) with the same model
used as both knowledge generator and inference model.

### Phase 1 — knowledge generation
- Exactly 3 knowledge calls per Eval300 sample.
- Every knowledge-generation prompt contains the first 3 spreadsheet
  demonstrations from the exact same subject.
- Each demonstration supplies its image(s), question, and reference `knowledge` field from `gkp_30_subjects_from_cot_completed_149.xlsx`.
- The target knowledge-generation input supplies only its image(s) and question; answer choices and gold answers are not sent in this phase.
- Knowledge calls use temperature `1.0`, nucleus
  `top_p=0.5`, seeds `[42, 43, 44]`, max 64 tokens, and stop
  at newline.
- Generated facts are checkpointed before any answer-inference phase.

### Phase 2 — zero-shot knowledge integration
- No demonstration is sent to the model.
- Each of the 3 frozen f

## Suggested Methods wording

We evaluated InternVL3.5-8B Q4_K_M using a multimodal adaptation of Generated Knowledge
Prompting. For each frozen MMMU-Pro query, the knowledge-generation prompt
contained the first three demonstrations of the same academic subject from
`gkp_30_subjects_from_cot_completed_149.xlsx`. Each demonstration contained its
source image(s), multiple-choice input, and a reference knowledge statement,
but no demonstration answer was used for final inference. The target input
contained no gold answer.

The same knowledge prompt was sampled three times to obtain three knowledge
statements. Knowledge generation used nucleus sampling with `top_p=0.5`,
model-specific temperature `1.0`, maximum 64 generated
tokens, newline termination, and fixed seeds 42/43/44 for reproducibility.

In a separate session, each frozen knowledge statement was concatenated with
the corresponding target image(s), question, and options. This integration
stage was zero-shot: no demonstrations were provided. Inference used
temperature 0, seed 42, no explicit top-p/top-k, and the same 8192-token output
cap as the zero-shot baseline. The model was asked to emit only one option
label, and generated-token log probabilities were requested from the local
llama.cpp OpenAI-compatible endpoint.

For answer option `a`, the aggregate GKP score was the maximum raw option-token
probability over the three knowledge-conditioned calls,
`max_m p(a | q, k_m)`. The final prediction was the option maximizing this
value. When the returned top-logprob list did not contain every legal option,
missing probabilities were not fabricated; such samples were excluded from
headline exact-logprob coverage and a greedy-label fallback was retained only
as a diagnostic.

Model/projector/runtime bytes were SHA-256 verified. The llama binary was never
executed directly from `/kaggle/input`; an exact copy under `/kaggle/temp` was
used after executable-bit, CUDA-driver, shared-library, and SHA checks.
